# CS570 — Project Deliverable 1: Data Loading & Exploratory Analysis
**Team:** Pentanet  
**Members:** AZATBEK ISMAILOV, FSEHAYE MEDHANIE ,MIR AHMAD ALI , MIR AHMAD ALI , RAMESH MANDAMANEDI , YUEXUAN LU  
**Date:** February 25, 2026


In [1]:
import os

# ── CHANGE THIS to where your ml-1m files are ──────────────────
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'data', 'raw')
# ───────────────────────────────────────────────────────────────

RATINGS_PATH = os.path.join(DATA_DIR, 'ratings.dat')
USERS_PATH   = os.path.join(DATA_DIR, 'users.dat')
MOVIES_PATH  = os.path.join(DATA_DIR, 'movies.dat')

for path in [RATINGS_PATH, USERS_PATH, MOVIES_PATH]:
    status = 'found' if os.path.exists(path) else 'NOT FOUND'
    print(f'{status}: {path}')




found: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\notebooks\..\data\raw\ratings.dat
found: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\notebooks\..\data\raw\users.dat
found: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\notebooks\..\data\raw\movies.dat


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('CS570-D1-MovieLens')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '2g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)


Spark version: 3.5.0


## 1. Data Loading
Each file is loaded with an **explicit schema** using `StructType`/`StructField`. No `inferSchema=True`.


In [3]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, LongType, StringType, FloatType
)

RATINGS_SCHEMA = StructType([
    StructField('UserID',    IntegerType(), nullable=False),
    StructField('MovieID',   IntegerType(), nullable=False),
    StructField('Rating',    FloatType(),   nullable=False),
    StructField('Timestamp', LongType(),    nullable=False),
])

USERS_SCHEMA = StructType([
    StructField('UserID',     IntegerType(), nullable=False),
    StructField('Gender',     StringType(),  nullable=False),
    StructField('Age',        IntegerType(), nullable=False),
    StructField('Occupation', IntegerType(), nullable=False),
    StructField('ZipCode',    StringType(),  nullable=True),
])

MOVIES_SCHEMA = StructType([
    StructField('MovieID', IntegerType(), nullable=False),
    StructField('Title',   StringType(),  nullable=False),
    StructField('Genres',  StringType(),  nullable=False),
])
print('Schemas defined.')


Schemas defined.


In [4]:
# ratings.dat
ratings = spark.read.option('sep', '::').schema(RATINGS_SCHEMA).csv(RATINGS_PATH)
ratings.printSchema()
print('Row count:', ratings.count())
ratings.show(5)
print(f"Columns: {ratings.columns}")

root
 |-- UserID: integer (nullable = true)
 |-- MovieID: integer (nullable = true)
 |-- Rating: float (nullable = true)
 |-- Timestamp: long (nullable = true)

Row count: 1000209
+------+-------+------+---------+
|UserID|MovieID|Rating|Timestamp|
+------+-------+------+---------+
|     1|   1193|   5.0|978300760|
|     1|    661|   3.0|978302109|
|     1|    914|   3.0|978301968|
|     1|   3408|   4.0|978300275|
|     1|   2355|   5.0|978824291|
+------+-------+------+---------+
only showing top 5 rows

Columns: ['UserID', 'MovieID', 'Rating', 'Timestamp']


In [5]:
# users.dat
users = spark.read.option('sep', '::').schema(USERS_SCHEMA).csv(USERS_PATH)
users.printSchema()
print('Row count:', users.count())
users.show(5)
print(f"Columns: {users.columns}")

root
 |-- UserID: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- ZipCode: string (nullable = true)

Row count: 6040
+------+------+---+----------+-------+
|UserID|Gender|Age|Occupation|ZipCode|
+------+------+---+----------+-------+
|     1|     F|  1|        10|  48067|
|     2|     M| 56|        16|  70072|
|     3|     M| 25|        15|  55117|
|     4|     M| 45|         7|  02460|
|     5|     M| 25|        20|  55455|
+------+------+---+----------+-------+
only showing top 5 rows

Columns: ['UserID', 'Gender', 'Age', 'Occupation', 'ZipCode']


In [6]:
# movies.dat
movies = spark.read.option('sep', '::').schema(MOVIES_SCHEMA).csv(MOVIES_PATH)
movies.printSchema()
print('Row count:', movies.count())
movies.show(5)
print(f"Columns: {movies.columns}")

root
 |-- MovieID: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)

Row count: 3883
+-------+--------------------+--------------------+
|MovieID|               Title|              Genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Animation|Childre...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|        Comedy|Drama|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows

Columns: ['MovieID', 'Title', 'Genres']


## 2. Join the Tables
`ratings` ↔ `users` on **UserID** · `ratings` ↔ `movies` on **MovieID** · both `inner` joins.


In [7]:
joined = (
    ratings
    .join(users,  on='UserID',  how='inner')
    .join(movies, on='MovieID', how='inner')
)

print('Row count:   ', joined.count())
print('Column count:', len(joined.columns))
joined.printSchema()
joined.show(5)
print(f"Columns: {joined.columns}")


Row count:    1000209
Column count: 10
root
 |-- MovieID: integer (nullable = true)
 |-- UserID: integer (nullable = true)
 |-- Rating: float (nullable = true)
 |-- Timestamp: long (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- ZipCode: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)

+-------+------+------+---------+------+---+----------+-------+--------------------+--------------------+
|MovieID|UserID|Rating|Timestamp|Gender|Age|Occupation|ZipCode|               Title|              Genres|
+-------+------+------+---------+------+---+----------+-------+--------------------+--------------------+
|   1193|     1|   5.0|978300760|     F|  1|        10|  48067|One Flew Over the...|               Drama|
|    661|     1|   3.0|978302109|     F|  1|        10|  48067|James and the Gia...|Animation|Childre...|
|    914|     1|   3.0|978301968|     F

## 3. Basic Statistics


In [10]:
joined.describe().show()


+-------+------------------+------------------+------------------+--------------------+-------+------------------+-----------------+------------------+--------------------+-------+
|summary|           MovieID|            UserID|            Rating|           Timestamp| Gender|               Age|       Occupation|           ZipCode|               Title| Genres|
+-------+------------------+------------------+------------------+--------------------+-------+------------------+-----------------+------------------+--------------------+-------+
|  count|           1000209|           1000209|           1000209|             1000209|1000209|           1000209|          1000209|           1000209|             1000209|1000209|
|   mean|1865.5398981612843| 3024.512347919285| 3.581564453029317| 9.722436954046655E8|   NULL| 29.73831369243828|8.036138447064564| 223239.8917114074|                NULL|   NULL|
| stddev|1096.0406894572482|1728.4126948999715|1.1171018453732606|1.2152558939916052E7|   NULL|

### Observations

The rating range is **1.0 to 5.0** (integer-only, no half-stars), with a mean of **3.58** and a standard deviation of **1.12**. 
The mean being well above the neutral midpoint of 3.0 reveals a **positive rating bias** — users are more likely to rate movies they enjoyed, which is a classic self-selection effect in recommender system datasets. 
The `Timestamp` column stands out as unusual: its mean (≈ 9.72 × 10⁸) and standard deviation (≈ 1.22 × 10⁷) are raw Unix epoch values that appear as large, unreadable numbers in the `describe()` output — these need to be converted to human-readable dates (the data spans April 2000 to February 2003) before any meaningful temporal analysis can be done.



## 4. EDA Questions


In [11]:
# A. Unique genres (explode pipe-separated values)
from pyspark.sql import functions as F

unique_genres = (
    joined
    .select(F.explode(F.split(F.col('Genres'), '\\|')).alias('genre'))
    .distinct()
    .count()
)
print(f'A. Unique individual genres: {unique_genres}')


A. Unique individual genres: 18


In [12]:
# B. Average rating — age group 25-34 (Age code = 25)
avg_25_34 = (
    joined
    .filter(F.col('Age') == 25)
    .agg(F.round(F.avg('Rating'), 2).alias('avg_rating'))
    .collect()[0]['avg_rating']
)
print(f'B. Average rating (25-34 age group): {avg_25_34}')


B. Average rating (25-34 age group): 3.55


In [13]:
# C. Movie with the most ratings
top = (
    joined
    .groupBy('MovieID', 'Title')
    .agg(F.count('*').alias('rating_count'))
    .orderBy(F.desc('rating_count'))
    .first()
)
print(f'C. Most rated movie : {top["Title"]}')
print(f'   Number of ratings: {top["rating_count"]}')


C. Most rated movie : American Beauty (1999)
   Number of ratings: 3428


#### EDA  — Rating Distribution

In [18]:
total = joined.count()
rating_dist = (
    joined
    .groupBy('Rating')
    .agg(F.count('*').alias('Count'))
    .withColumn('Percentage', F.round(F.col('Count') / total * 100, 1))
    .withColumn('Bar', F.expr(f"repeat('█', CAST(Count / {total} * 50 AS INT))"))
    .orderBy('Rating')
)
rating_dist.show(truncate=False)


+------+------+----------+-----------------+
|Rating|Count |Percentage|Bar              |
+------+------+----------+-----------------+
|1.0   |56174 |5.6       |██               |
|2.0   |107557|10.8      |█████            |
|3.0   |261197|26.1      |█████████████    |
|4.0   |348971|34.9      |█████████████████|
|5.0   |226310|22.6      |███████████      |
+------+------+----------+-----------------+



#### EDA  — Top 10 Highest-Rated Movies (≥ 100 ratings)
Filtering by minimum 100 ratings avoids obscure films with a handful of perfect scores.

In [19]:
# Top 10 highest-rated movies with statistical significance
top_rated = (
    joined
    .groupBy('MovieID', 'Title')
    .agg(
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
        F.count('*').alias('Num_Ratings'),
    )
    .filter(F.col('Num_Ratings') >= 100)
    .orderBy(F.desc('Avg_Rating'))
)
top_rated.show(10, truncate=False)

+-------+-------------------------------------------------------------------+----------+-----------+
|MovieID|Title                                                              |Avg_Rating|Num_Ratings|
+-------+-------------------------------------------------------------------+----------+-----------+
|2019   |Seven Samurai (The Magnificent Seven) (Shichinin no samurai) (1954)|4.56      |628        |
|318    |Shawshank Redemption, The (1994)                                   |4.55      |2227       |
|50     |Usual Suspects, The (1995)                                         |4.52      |1783       |
|858    |Godfather, The (1972)                                              |4.52      |2223       |
|745    |Close Shave, A (1995)                                              |4.52      |657        |
|1148   |Wrong Trousers, The (1993)                                         |4.51      |882        |
|527    |Schindler's List (1993)                                            |4.51      |230

#### EDA  — Gender Rating Patterns
Do male and female users rate differently?

In [20]:
# Average rating by gender + volume
gender_stats = (
    joined
    .groupBy('Gender')
    .agg(
        F.count('*').alias('Total_Ratings'),
        F.round(F.avg('Rating'), 3).alias('Avg_Rating'),
        F.round(F.stddev('Rating'), 3).alias('Std_Rating'),
        F.countDistinct('UserID').alias('Unique_Users'),
    )
    .withColumn('Ratings_Per_User', F.round(F.col('Total_Ratings') / F.col('Unique_Users'), 1))
    .orderBy('Gender')
)
gender_stats.show(truncate=False)


+------+-------------+----------+----------+------------+----------------+
|Gender|Total_Ratings|Avg_Rating|Std_Rating|Unique_Users|Ratings_Per_User|
+------+-------------+----------+----------+------------+----------------+
|F     |246440       |3.62      |1.111     |1709        |144.2           |
|M     |753769       |3.569     |1.119     |4331        |174.0           |
+------+-------------+----------+----------+------------+----------------+



#### EDA  — Age Group Rating Behavior
Age codes: 1=Under 18, 18=18-24, 25=25-34, 35=35-44, 45=45-49, 50=50-55, 56=56+


In [21]:
# Rating behavior by age group
age_labels = {1:'Under 18', 18:'18-24', 25:'25-34', 35:'35-44', 45:'45-49', 50:'50-55', 56:'56+'}
from pyspark.sql.functions import create_map, lit
mapping = create_map([val for k, v in age_labels.items() for val in (lit(k), lit(v))])

age_stats = (
    joined
    .withColumn('Age_Group', mapping[F.col('Age')])
    .groupBy('Age', 'Age_Group')
    .agg(
        F.countDistinct('UserID').alias('Users'),
        F.count('*').alias('Total_Ratings'),
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
    )
    .withColumn('Ratings_Per_User', F.round(F.col('Total_Ratings') / F.col('Users'), 1))
    .orderBy('Age')
)
age_stats.show(truncate=False)


+---+---------+-----+-------------+----------+----------------+
|Age|Age_Group|Users|Total_Ratings|Avg_Rating|Ratings_Per_User|
+---+---------+-----+-------------+----------+----------------+
|1  |Under 18 |222  |27211        |3.55      |122.6           |
|18 |18-24    |1103 |183536       |3.51      |166.4           |
|25 |25-34    |2096 |395556       |3.55      |188.7           |
|35 |35-44    |1193 |199003       |3.62      |166.8           |
|45 |45-49    |550  |83633        |3.64      |152.1           |
|50 |50-55    |496  |72490        |3.71      |146.1           |
|56 |56+      |380  |38780        |3.77      |102.1           |
+---+---------+-----+-------------+----------+----------------+



#### EDA  — Genre Popularity vs Quality
Which genres are most watched vs. most loved?


In [22]:
# Genre analysis: popularity (count) vs quality (avg rating)
genre_stats = (
    joined
    .select(F.explode(F.split(F.col('Genres'), '\\|')).alias('Genre'), 'Rating')
    .groupBy('Genre')
    .agg(
        F.count('*').alias('Num_Ratings'),
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
        F.round(F.stddev('Rating'), 2).alias('Std_Rating'),
    )
    .orderBy(F.desc('Num_Ratings'))
)
genre_stats.show(20, truncate=False)


+-----------+-----------+----------+----------+
|Genre      |Num_Ratings|Avg_Rating|Std_Rating|
+-----------+-----------+----------+----------+
|Comedy     |356580     |3.52      |1.12      |
|Drama      |354529     |3.77      |1.05      |
|Action     |257457     |3.49      |1.13      |
|Thriller   |189680     |3.57      |1.11      |
|Sci-Fi     |157294     |3.47      |1.16      |
|Romance    |147523     |3.61      |1.07      |
|Adventure  |133953     |3.48      |1.13      |
|Crime      |79541      |3.71      |1.08      |
|Horror     |76386      |3.22      |1.23      |
|Children's |72186      |3.42      |1.16      |
|War        |68527      |3.89      |1.07      |
|Animation  |43293      |3.68      |1.08      |
|Musical    |41533      |3.67      |1.1       |
|Mystery    |40178      |3.67      |1.09      |
|Fantasy    |36301      |3.45      |1.13      |
|Western    |20683      |3.64      |1.1       |
|Film-Noir  |18261      |4.08      |0.93      |
|Documentary|7910       |3.93      |1.03

#### EDA  — User Activity Distribution
Is there a power-law pattern in user engagement?


In [23]:
# User activity — classify into engagement tiers
user_activity = ratings.groupBy('UserID').agg(F.count('*').alias('num_ratings'))

user_tiers = (
    user_activity
    .withColumn('Tier', F.when(F.col('num_ratings') < 50, 'Light (< 50)')
                         .when(F.col('num_ratings') < 150, 'Medium (50-149)')
                         .when(F.col('num_ratings') < 500, 'Active (150-499)')
                         .otherwise('Power (500+)'))
    .groupBy('Tier')
    .agg(
        F.count('*').alias('Users'),
        F.sum('num_ratings').alias('Total_Ratings'),
        F.round(F.avg('num_ratings'), 1).alias('Avg_Ratings_Per_User'),
    )
    .orderBy('Avg_Ratings_Per_User')
)
user_tiers.show(truncate=False)

# Quick stats
print('User activity summary:')
user_activity.select(
    F.min('num_ratings').alias('Min'),
    F.expr('percentile_approx(num_ratings, 0.25)').alias('Q1'),
    F.expr('percentile_approx(num_ratings, 0.5)').alias('Median'),
    F.expr('percentile_approx(num_ratings, 0.75)').alias('Q3'),
    F.max('num_ratings').alias('Max'),
    F.round(F.avg('num_ratings'), 1).alias('Mean'),
).show(truncate=False)


+----------------+-----+-------------+--------------------+
|Tier            |Users|Total_Ratings|Avg_Ratings_Per_User|
+----------------+-----+-------------+--------------------+
|Light (< 50)    |1743 |56738        |32.6                |
|Medium (50-149) |2201 |199399       |90.6                |
|Active (150-499)|1697 |453763       |267.4               |
|Power (500+)    |399  |290309       |727.6               |
+----------------+-----+-------------+--------------------+

User activity summary:
+---+---+------+---+----+-----+
|Min|Q1 |Median|Q3 |Max |Mean |
+---+---+------+---+----+-----+
|20 |44 |95    |207|2314|165.6|
+---+---+------+---+----+-----+



#### EDA  — Rating Trends Over Time
How does rating volume and average change over the data collection period?


In [24]:
# Monthly rating trends
temporal = (
    joined
    .withColumn('date', F.from_unixtime('Timestamp'))
    .withColumn('YearMonth', F.date_format('date', 'yyyy-MM'))
    .groupBy('YearMonth')
    .agg(
        F.count('*').alias('Num_Ratings'),
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
        F.countDistinct('UserID').alias('Active_Users'),
    )
    .orderBy('YearMonth')
)
temporal.show(50, truncate=False)


+---------+-----------+----------+------------+
|YearMonth|Num_Ratings|Avg_Rating|Active_Users|
+---------+-----------+----------+------------+
|2000-04  |11672      |3.57      |89          |
|2000-05  |67827      |3.61      |486         |
|2000-06  |55146      |3.64      |510         |
|2000-07  |93640      |3.62      |797         |
|2000-08  |178129     |3.58      |1289        |
|2000-09  |53043      |3.62      |571         |
|2000-10  |42165      |3.61      |504         |
|2000-11  |291012     |3.57      |2359        |
|2000-12  |112258     |3.58      |1233        |
|2001-01  |18302      |3.54      |543         |
|2001-02  |7953       |3.54      |393         |
|2001-03  |5854       |3.55      |323         |
|2001-04  |5194       |3.47      |289         |
|2001-05  |4987       |3.47      |276         |
|2001-06  |4930       |3.47      |270         |
|2001-07  |4728       |3.47      |285         |
|2001-08  |4565       |3.46      |246         |
|2001-09  |2975       |3.55      |193   

## 5. Data Quality Observations


In [15]:
# Issue 3: Raw Timestamp — hard to read
joined.agg(F.min('Timestamp'), F.max('Timestamp')).show()
joined.select(F.from_unixtime('Timestamp').alias('readable_date')).show(5)


+--------------+--------------+
|min(Timestamp)|max(Timestamp)|
+--------------+--------------+
|     956703932|    1046454590|
+--------------+--------------+

+-------------------+
|      readable_date|
+-------------------+
|2000-12-31 14:12:40|
|2000-12-31 14:35:09|
|2000-12-31 14:32:48|
|2000-12-31 14:04:35|
|2001-01-06 15:38:11|
+-------------------+
only showing top 5 rows



In [25]:
# Issue 2: Non-standard Zip Codes (6-digit and 9-digit codes)
total_users = users.count()

# Find zip codes that are NOT exactly 5 digits
non_standard = users.filter(~F.col('ZipCode').rlike(r'^\d{5}$'))
non_standard_count = non_standard.count()

# Classify by length
non_standard_with_len = non_standard.withColumn('zip_length', F.length('ZipCode'))

print(f'Total users:                {total_users:,}')
print(f'Non-standard zip codes:     {non_standard_count}')
print()

# Show breakdown by zip code length
print('Breakdown by zip code length:')
non_standard_with_len.groupBy('zip_length').count().orderBy('zip_length').show()

# Specifically highlight 6-digit and 9-digit codes
print('6-digit zip codes:')
non_standard.filter(F.length('ZipCode') == 6).select('UserID', 'ZipCode').show(truncate=False)

print('9-digit zip codes:')
non_standard.filter(F.length('ZipCode') == 9).select('UserID', 'ZipCode').show(truncate=False)


Total users:                6,040
Non-standard zip codes:     81

Breakdown by zip code length:
+----------+-----+
|zip_length|count|
+----------+-----+
|         6|   11|
|         7|    3|
|         9|    1|
|        10|   66|
+----------+-----+

6-digit zip codes:
+------+-------+
|UserID|ZipCode|
+------+-------+
|1091  |345567 |
|1434  |495321 |
|2106  |495321 |
|2853  |444555 |
|3355  |400060 |
|3905  |361069 |
|4454  |111225 |
|4913  |970025 |
|4973  |949702 |
|5510  |191004 |
|5904  |954025 |
+------+-------+

9-digit zip codes:
+------+---------+
|UserID|ZipCode  |
+------+---------+
|5100  |193122042|
+------+---------+



In [26]:
# Issue 3: Movies with zero ratings (orphan movies)
# Count ratings per movie
rating_counts = ratings.groupBy('MovieID').agg(F.count('*').alias('Ratings'))

# Left join movies with rating counts — orphans will have null Ratings
movies_with_counts = movies.join(rating_counts, on='MovieID', how='left').fillna(0, subset=['Ratings'])

# Filter orphan movies (zero ratings)
orphan_movies = movies_with_counts.filter(F.col('Ratings') == 0)
orphan_count = orphan_movies.count()
total_movies = movies.count()

print(f'Total movies in dataset:       {total_movies:,}')
print(f'Movies with zero ratings:      {orphan_count}')
print(f'Movies with at least 1 rating: {total_movies - orphan_count:,}')
print()
print('Sample orphan movies (no ratings):')
orphan_movies.select('MovieID', 'Title', 'Genres', 'Ratings').orderBy('MovieID').show(10, truncate=False)


Total movies in dataset:       3,883
Movies with zero ratings:      177
Movies with at least 1 rating: 3,706

Sample orphan movies (no ratings):
+-------+-----------------------------------+---------------------+-------+
|MovieID|Title                              |Genres               |Ratings|
+-------+-----------------------------------+---------------------+-------+
|51     |Guardian Angel (1994)              |Action|Drama|Thriller|0      |
|109    |Headless Body in Topless Bar (1995)|Comedy               |0      |
|115    |Happiness Is in the Field (1995)   |Comedy               |0      |
|143    |Gospa (1995)                       |Drama                |0      |
|284    |New York Cop (1996)                |Action|Crime         |0      |
|285    |Beyond Bedlam (1993)               |Drama|Horror         |0      |
|395    |Desert Winds (1995)                |Drama                |0      |
|399    |Girl in the Cadillac (1995)        |Drama                |0      |
|400    |Homage (19

### Issues Found

**Issue 1 — Raw Unix timestamps are not human-readable.**  
The `Timestamp` column stores ratings as raw Unix epoch integers (e.g., `978300760`), which are not interpretable at a glance. The timestamps range from **956,703,932** (April 25, 2000) to **1,046,454,590** (February 28, 2003). While the values are valid and contain no negatives or zeros, they need to be converted to proper datetime format for any time-based analysis such as trend detection or seasonal patterns.  
- Handle in D2: Convert the `Timestamp` column to a readable datetime using `F.from_unixtime('Timestamp')` and extract useful features such as year, month, day of week, and hour for temporal analysis.

**Issue 2 — Non-standard zip code formats (6-digit and 9-digit codes).**  
Standard US zip codes are exactly 5 digits (e.g., `48067`). Our analysis found entries with **6 digits** (e.g., `111225`) and **9 digits** (e.g., `193122042`) that do not correspond to any valid US postal format. These are likely **data entry errors** — for example, a user may have accidentally typed an extra digit, or concatenated a ZIP+4 code without the dash. This is a problem because:  
1. These zip codes **cannot be mapped to real geographic locations**, making location-based analysis unreliable.  
2. They will **fail to join** with any external geographic lookup table (e.g., zip-to-state mapping), causing data loss.  
3. If used in grouping or aggregation, they will create **incorrect or orphan categories** that skew results.  
- Handle in D2: Truncate all zip codes to the first 5 characters using `F.substring('ZipCode', 1, 5)`, or flag the malformed entries and exclude them from geographic analysis.

**Issue 3 — 177 movies in the catalog have zero ratings.**  
By left-joining the movies table with a per-movie rating count, we found that **177 movies** have a `Ratings` count of **0** — they exist in the catalog but have never been rated by any user. These orphan records are problematic because:  
1. They are **unusable for collaborative filtering**, since the algorithm requires at least some user–item interactions to generate recommendations.  
2. They **inflate the item space** unnecessarily, increasing computation without adding predictive value.  
3. They could introduce **cold-start bias** if included in evaluation metrics, making model performance appear worse than it is.  
- Handle in D2: Filter out movies with zero ratings before model training, or flag them separately for a cold-start handling strategy.


## 6. Contribution Statement

**AZATBEK ISMAILOV:** I contributed to the data quality observations section (Section 5). I identified three key issues in the dataset: (1) raw Unix timestamps that are not human-readable and need conversion for temporal analysis, (2) non-standard zip code formats including 6-digit and 9-digit codes that are likely data entry errors and would break geographic lookups, and (3) 177 orphan movies in the catalog with zero ratings that are unusable for collaborative filtering. For each issue, I wrote the PySpark queries to detect and quantify the problem, and proposed handling strategies for Deliverable 2.

**FSEHAYE MEDHANIE:** I contributed to the data loading and table join (Sections 1, 2). For Section 1, I defined explicit schemas using StructType and StructField for all three source files — assigning FloatType for Rating, LongType for Timestamp, and nullable=True only for ZipCode which the dataset documentation marks as voluntary — and verified the row counts: 1,000,209 ratings, 6,040 users, and 3,883 movies after join filtering. For Section 2, I performed the three-table join by chaining two inner joins — ratings to users on UserID, then to movies on MovieID — and confirmed the result produces 1,000,209 rows across 10 columns with no row loss, meaning every rating has a matched user and movie. From the other pair I learned how F.expr() can embed Python variables into Spark SQL strings to work around the Column-not-iterable limitation of functions like F.repeat().

**MIR AHMAD ALI:** I contributed...

**RAMESH MANDAMANEDI:** I contributed...

**YUEXUAN LU:** I contributed for completing Part 4 (Exploratory Data Analysis), including data cleaning, visualization, and answering all EDA questions. Through my partner’s work, I gained a better understanding of how modeling techniques build upon exploratory analysis and how different components of a data project connect together. This collaboration helped me see the full data analysis pipeline more clearly.
